# TimeSeries — PyArrow Backend

The `TimeSeries` class is a PyArrow-backed container for time series data. Unlike `TimeSeriesList`, which stores data in Python lists, `TimeSeries` keeps everything in a `pyarrow.Table` — making it ideal for database integration via Arrow Flight / ADBC cursors.

Key characteristics:

- **Four data shapes**: SIMPLE, VERSIONED, CORRECTED, AUDIT — matching the read patterns of a bi-temporal database
- **Arrow-native**: data arrives from a cursor as `pa.Table` and stays that way; no redundant copies
- **Rich metadata**: name, unit, data type, geographic location, timezone, frequency, and arbitrary labels
- **Pandas interop**: `from_pandas()` / `to_pandas()` with automatic index handling per shape

## Installation

```bash
pip install timedatamodel[pyarrow]
# or with all extras
pip install timedatamodel[all]
```

In [1]:
import pandas as pd
import pyarrow as pa

import timedatamodel as tdm
from timedatamodel import TimeSeries, DataShape
from timedatamodel.enums import DataType, TimeSeriesType
from timedatamodel.location import GeoLocation

## Quick start — SIMPLE shape

The most common shape is **SIMPLE**: one observation per `valid_time`. Use `TimeSeries.from_pandas()` to create one from a DataFrame.

In [2]:
df_simple = pd.DataFrame({
    "valid_time": pd.date_range("2024-01-15", periods=24, freq="1h", tz="UTC"),
    "value": [
        120.0, 115.0, 108.0, 105.0, 102.0, 100.0,
        110.0, 135.0, 160.0, 175.0, 180.0, 178.0,
        172.0, 170.0, 168.0, 165.0, 175.0, 190.0,
        200.0, 195.0, 180.0, 165.0, 145.0, 130.0,
    ],
})

ts = TimeSeries.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency="1h",
    data_type=DataType.OBSERVATION,
    location=GeoLocation(latitude=57.7, longitude=11.97),
    timezone="Europe/Stockholm",
    description="Hourly wind power output from farm Alpha",
)
ts

Name,wind_power
Shape,SIMPLE
Rows,24
Frequency,1h
Timezone,Europe/Stockholm
Unit,MW
Data type,OBSERVATION
Location,"57.7°N, 11.97°E"
Description,Hourly wind power output from farm Alpha
valid_time,wind_power
2024-01-15 00:00,120.0


The `valid_time` column can also live in the DataFrame index — `from_pandas()` handles both layouts:

In [3]:
df_indexed = df_simple.set_index("valid_time")
ts_from_index = TimeSeries.from_pandas(df_indexed, name="wind_power", unit="MW")

print(f"Shape: {ts_from_index.shape}")
print(f"Rows:  {ts_from_index.num_rows}")

Shape: DataShape.SIMPLE
Rows:  24


## Key properties

| Property | Description |
|---|---|
| `shape` | `DataShape` enum — which temporal columns are present |
| `num_rows` | Number of rows in the Arrow table |
| `columns` | List of column names |
| `table` | The underlying `pyarrow.Table` (read-only by convention) |

In [4]:
print(f"shape:    {ts.shape}")
print(f"num_rows: {ts.num_rows}")
print(f"columns:  {ts.columns}")
print(f"len():    {len(ts)}")
print()
print("Underlying Arrow schema:")
print(ts.table.schema)

shape:    DataShape.SIMPLE
num_rows: 24
columns:  ['valid_time', 'value']
len():    24

Underlying Arrow schema:
valid_time: timestamp[us, tz=UTC]
value: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 350


## The four data shapes

`DataShape` is inferred automatically from the column names in the Arrow table.

| Shape | Temporal columns | Use case |
|---|---|---|
| **SIMPLE** | `valid_time` | Flat, immutable observations |
| **VERSIONED** | `knowledge_time`, `valid_time` | Forecast revision history |
| **CORRECTED** | `valid_time`, `change_time` | Corrected meter readings (no forecast) |
| **AUDIT** | `knowledge_time`, `change_time`, `valid_time` | Full bi-temporal log |

### VERSIONED shape

A `VERSIONED` series stores the full forecast revision history — one row per `(knowledge_time, valid_time)` pair. This is the standard output of a forecast-issuing system where new runs update future values.

In [5]:
# Two forecast runs (knowledge_times), each covering 6 hours ahead
rows = []
for kt_offset, values in [
    (0, [100.0, 105.0, 110.0, 108.0, 103.0, 98.0]),
    (6, [102.0, 107.0, 112.0, 111.0, 106.0, 101.0]),
]:
    kt = pd.Timestamp("2024-01-15 00:00", tz="UTC") + pd.Timedelta(hours=kt_offset)
    for h, v in enumerate(values):
        rows.append({
            "knowledge_time": kt,
            "valid_time": pd.Timestamp("2024-01-15 00:00", tz="UTC") + pd.Timedelta(hours=h),
            "value": v,
        })

df_versioned = pd.DataFrame(rows)

ts_v = TimeSeries.from_pandas(
    df_versioned,
    name="wind_power_forecast",
    unit="MW",
    frequency="1h",
    data_type=DataType.FORECAST,
    timeseries_type=TimeSeriesType.OVERLAPPING,
)
ts_v

TimeSeries
┌───────────────────────────────────────────────────────────┐
│  Name:             wind_power_forecast                    │
│  Shape:            VERSIONED                              │
│  Rows:             12                                     │
│  Frequency:        1h                                     │
│  Timezone:         UTC                                    │
│  Unit:             MW                                     │
│  Data type:        FORECAST                               │
│  Timeseries type:  OVERLAPPING                            │
├───────────────────────────────────────────────────────────┤
│                                      wind_power_forecast  │
│  2024-01-15 00:00, 2024-01-15 00:00                100.0  │
│  2024-01-15 00:00, 2024-01-15 01:00                105.0  │
│  2024-01-15 00:00, 2024-01-15 02:00                110.0  │
│  ...                                                 ...  │
│  2024-01-15 06:00, 2024-01-15 03:00                111.0  │
│  2024-01-15 06:00, 2024-01-15 04:00                106.0  │
│  2024-01-15 06:00, 2024-01-15 05:00                101.0  │
└───────────────────────────────────────────────────────────┘

### CORRECTED shape

A `CORRECTED` series captures historical corrections to flat (non-forecast) data — for example, when sensor readings are recalibrated. Each correction adds a `change_time` column recording when the edit was made. CORRECTED shapes can only be created via `from_arrow()` (read from a database), not `from_pandas()`.

In [6]:
ts_type = pa.timestamp("us", tz="UTC")

corrected_table = pa.table({
    "valid_time":  pa.array([
        pd.Timestamp("2024-01-15 00:00", tz="UTC"),
        pd.Timestamp("2024-01-15 01:00", tz="UTC"),
        pd.Timestamp("2024-01-15 01:00", tz="UTC"),  # corrected
    ], type=ts_type),
    "change_time": pa.array([
        pd.Timestamp("2024-01-20 09:00", tz="UTC"),
        pd.Timestamp("2024-01-20 09:00", tz="UTC"),
        pd.Timestamp("2024-01-22 14:30", tz="UTC"),  # later correction
    ], type=ts_type),
    "value":       pa.array([120.0, 115.0, 116.5]),  # 116.5 corrects 115.0
})

ts_c = TimeSeries.from_arrow(
    corrected_table,
    name="meter_reading",
    unit="MWh",
    data_type=DataType.OBSERVATION,
)
ts_c

TimeSeries
┌─────────────────────────────────────────────────────┐
│  Name:             meter_reading                    │
│  Shape:            CORRECTED                        │
│  Rows:             3                                │
│  Timezone:         UTC                              │
│  Unit:             MWh                              │
│  Data type:        OBSERVATION                      │
├─────────────────────────────────────────────────────┤
│                                      meter_reading  │
│  2024-01-15 00:00, 2024-01-20 09:00          120.0  │
│  2024-01-15 01:00, 2024-01-20 09:00          115.0  │
│  2024-01-15 01:00, 2024-01-22 14:30          116.5  │
└─────────────────────────────────────────────────────┘

## Creating from an Arrow table

`TimeSeries.from_arrow()` is the preferred path when data is already in Arrow format — for example, fetched directly from a database via ADBC:

```python
# Typical ADBC usage
cursor.execute("SELECT valid_time, value FROM series WHERE ...")
table = cursor.fetch_arrow_table()
ts = TimeSeries.from_arrow(table, name="wind_power", unit="MW")
```

All timestamp columns must already use `pa.timestamp('us', tz='UTC')`.

In [7]:
arrow_table = pa.table({
    "valid_time": pa.array(
        pd.date_range("2024-01-15", periods=6, freq="1h", tz="UTC"),
        type=pa.timestamp("us", tz="UTC"),
    ),
    "value": pa.array([100.0, 105.0, None, 108.0, 103.0, 98.0]),
})

ts_arrow = TimeSeries.from_arrow(
    arrow_table,
    name="load",
    unit="MW",
    frequency="1h",
    data_type=DataType.OBSERVATION,
)
ts_arrow

Name,load
Shape,SIMPLE
Rows,6
Frequency,1h
Timezone,UTC
Unit,MW
Data type,OBSERVATION
valid_time,load
2024-01-15 00:00,100.0
2024-01-15 01:00,105.0
2024-01-15 02:00,NaN


## Coverage bar

A coverage bar shows which timesteps have values present (non-null) vs. missing. Call `.coverage_bar()` on any `TimeSeries` to display it.

In [8]:
ts_arrow.coverage_bar()

load  ██░███
      2024-01-15 00:00  2024-01-15 05:00

## Converting to pandas

`to_pandas()` restores the conventional index used by the existing read API. The index structure depends on the shape:

| Shape | Index |
|---|---|
| SIMPLE | `valid_time` |
| VERSIONED | `(knowledge_time, valid_time)` MultiIndex |
| CORRECTED | `(valid_time, change_time)` MultiIndex |
| AUDIT | `(knowledge_time, change_time, valid_time)` MultiIndex |

In [9]:
df_out = ts.to_pandas()
print(f"Index name: {df_out.index.name}")
df_out.head()

Index name: valid_time


,value
valid_time,
2024-01-15 00:00:00+00:00,120.0
2024-01-15 01:00:00+00:00,115.0
2024-01-15 02:00:00+00:00,108.0
2024-01-15 03:00:00+00:00,105.0
2024-01-15 04:00:00+00:00,102.0


In [10]:
df_v_out = ts_v.to_pandas()
print(f"Index names: {df_v_out.index.names}")
df_v_out.head(8)

Index names: ['knowledge_time', 'valid_time']


value
knowledge_time            valid_time                      
2024-01-15 00:00:00+00:00 2024-01-15 00:00:00+00:00  100.0
                          2024-01-15 01:00:00+00:00  105.0
                          2024-01-15 02:00:00+00:00  110.0
                          2024-01-15 03:00:00+00:00  108.0
                          2024-01-15 04:00:00+00:00  103.0
                          2024-01-15 05:00:00+00:00   98.0
2024-01-15 06:00:00+00:00 2024-01-15 00:00:00+00:00  102.0
                          2024-01-15 01:00:00+00:00  107.0

## Metadata

`metadata_dict()` returns all metadata fields as a plain dict — useful for serialisation or logging.

In [11]:
import json
print(json.dumps(ts.metadata_dict(), indent=2, default=str))

{
  "name": "wind_power",
  "description": "Hourly wind power output from farm Alpha",
  "unit": "MW",
  "labels": {},
  "timezone": "Europe/Stockholm",
  "frequency": "1h",
  "data_type": "OBSERVATION",
  "location": {
    "longitude": 11.97,
    "latitude": 57.7
  },
  "timeseries_type": "FLAT",
  "shape": "SIMPLE",
  "num_rows": 24
}


## Labels

Arbitrary key-value labels let you tag series for filtering and provenance. They appear in the repr when set.

In [12]:
ts_labelled = TimeSeries.from_pandas(
    df_simple,
    name="wind_power",
    unit="MW",
    frequency="1h",
    labels={"site": "Alpha", "turbine_model": "V150-4.5", "country": "SE"},
)
ts_labelled

Name,wind_power
Shape,SIMPLE
Rows,24
Frequency,1h
Timezone,UTC
Unit,MW
Labels,"{'site': 'Alpha', 'turbine_model': 'V150-4.5', 'country': 'SE'}"
valid_time,wind_power
2024-01-15 00:00,120.0
2024-01-15 01:00,115.0
2024-01-15 02:00,108.0


## Validating for database insert

`validate_for_insert()` confirms that the series can be written back to the database and returns `(pa.Table, DataShape)`. Only **SIMPLE** and **VERSIONED** shapes are insertable — **CORRECTED** and **AUDIT** shapes are read-only query results.

In [13]:
table, shape = ts.validate_for_insert()
print(f"Shape: {shape}")
print(f"Rows:  {table.num_rows}")
print(f"Ready to insert: ✓")

Shape: DataShape.SIMPLE
Rows:  24
Ready to insert: ✓


In [14]:
# CORRECTED shape raises ValueError — it is a read-only audit result
try:
    ts_c.validate_for_insert()
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: TimeSeries with shape CORRECTED cannot be inserted. Only SIMPLE and VERSIONED shapes are supported for insert.


## DataShape enum

`DataShape` is exported from `timedatamodel` and can be used to branch on the shape of an incoming series:

In [15]:
for series in [ts, ts_v, ts_c]:
    match series.shape:
        case DataShape.SIMPLE:
            print(f"{series.name!r:30s} → flat observation, {series.num_rows} rows")
        case DataShape.VERSIONED:
            print(f"{series.name!r:30s} → forecast history, {series.num_rows} rows")
        case DataShape.CORRECTED:
            print(f"{series.name!r:30s} → corrected readings, {series.num_rows} rows")
        case DataShape.AUDIT:
            print(f"{series.name!r:30s} → full audit log, {series.num_rows} rows")

'wind_power'                   → flat observation, 24 rows
'wind_power_forecast'          → forecast history, 12 rows
'meter_reading'                → corrected readings, 3 rows


## Summary

In this notebook you learned how to:

- Create a `TimeSeries` from a pandas DataFrame (`from_pandas`) or an Arrow table (`from_arrow`)
- Understand the four **data shapes** (SIMPLE, VERSIONED, CORRECTED, AUDIT) and when each is used
- Inspect key properties: `shape`, `num_rows`, `columns`, `table`
- Convert back to pandas with `to_pandas()`, which restores the correct index per shape
- Attach metadata: name, unit, data type, location, timezone, frequency, and arbitrary labels
- Validate a series for database insert with `validate_for_insert()`
- Branch on shape using `DataShape` and `match`